In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

# Estimate pooled subjects TRFs

In [2]:
def estimate_pooled_universal_trf(subjects, predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.BACKWARD, padded=False, predictor_dir=FUGLSANG_PREDICTOR_DIR):

    TRF_LAG_START        = -0.100
    TRF_LAG_END          =  1.000
    BASIS_FUNCTION_WIDTH =  0.050

    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]
    predictor_names = sorted(predictor_names, key=lambda p: p.value)

    model_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor_names, attention, model_type, generalised=GENERALISATION_TYPE.POOLED
    )
    suffix    = "_padded" if padded else ""
    save_path = FUGLSANG_GENERAL_TRF_DIR / f"full_{model_name}.pickle"

    if save_path.exists():
        print(f"TRF exists at {save_path}, skipping.")
        return

    # ----------------------------------------------------------------
    # Collect one continuous EEG segment and one predictor segment
    # per subject. Boosting will treat each subject as an independent
    # segment and estimate one TRF that jointly explains all of them.
    # ----------------------------------------------------------------
    all_eeg        = []  # one NDVar (sensor, time) per subject
    all_predictors = []  # one NDVar (time,) per subject, per predictor

    for subject in subjects:
        print(f"Loading {subject}...")

        eeg_path = FUGLSANG_EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"
        if not eeg_path.exists():
            print(f"  Missing EEG, skipping.")
            continue

        eeg = eelbrain.load.unpickle(eeg_path)

        failed     = False
        predictors = []
        for p in predictor_names:
            predictor_name = helper_functions.get_attentional_predictor_name(p, attention, padded)
            predictor_path = predictor_dir / subject / f"{predictor_name}_concat.pickle"

            if not predictor_path.exists():
                print(f"  Missing predictor {predictor_path}, skipping.")
                failed = True
                break

            predictors.append(eelbrain.load.unpickle(predictor_path))

        if failed:
            continue

        all_eeg.append(eeg)
        all_predictors.append(predictors)
        print(f"  Loaded: EEG={eeg.shape}, predictor={predictors[0].shape}")

    if not all_eeg:
        print("No data loaded, aborting.")
        return

    print("-" * 50)
    print(f"Estimating pooled {model_name} TRF across {len(all_eeg)} subjects...")

    # ----------------------------------------------------------------
    # Prepare stimulus
    # One predictor:  [pred_s1, pred_s2, ...]
    # Two predictors: [[pred1_s1, pred1_s2, ...], [pred2_s1, pred2_s2, ...]]
    # ----------------------------------------------------------------
    if len(predictor_names) == 1:
        stimulus = [subj_preds[0] for subj_preds in all_predictors]
    else:
        stimulus = [
            [all_predictors[s][p] for s in range(len(all_eeg))]
            for p in range(len(predictor_names))
        ]

    if model_type == MODEL_TYPE.FORWARD:
        tmin, tmax = TRF_LAG_START, TRF_LAG_END
        trf = eelbrain.boosting(
        eelbrain.concatenate(all_eeg),
        eelbrain.concatenate(stimulus) if len(predictor_names) == 1 else [eelbrain.concatenate(s) for s in stimulus],
        tmin, tmax,
        error='l1', basis=BASIS_FUNCTION_WIDTH,
        partitions=5, test=1, selective_stopping=True
    )
    else:
        tmin, tmax = -TRF_LAG_END, -TRF_LAG_START
        trf = eelbrain.boosting(
        eelbrain.concatenate(stimulus) if len(predictor_names) == 1 else [eelbrain.concatenate(s) for s in stimulus],
        eelbrain.concatenate(all_eeg),
        tmin, tmax,
        error='l1', basis=BASIS_FUNCTION_WIDTH,
        partitions=5, test=1, selective_stopping=True
    )

    eelbrain.save.pickle(trf, save_path)
    print(f"Saved → {save_path}")

In [3]:
SUBJECTS = helper_functions.fuglsang_get_subjects()

estimate_pooled_universal_trf(subjects=SUBJECTS, predictor_names=PREDICTOR_TYPE.ENVELOPE, attention=ATTENTION_TYPE.ATTENDED, model_type=MODEL_TYPE.BACKWARD)
estimate_pooled_universal_trf(subjects=SUBJECTS, predictor_names=PREDICTOR_TYPE.ENVELOPE, attention=ATTENTION_TYPE.IGNORED, model_type=MODEL_TYPE.BACKWARD)
estimate_pooled_universal_trf(subjects=SUBJECTS, predictor_names=PREDICTOR_TYPE.ENVELOPE_ONSET, attention=ATTENTION_TYPE.ATTENDED, model_type=MODEL_TYPE.BACKWARD)
estimate_pooled_universal_trf(subjects=SUBJECTS, predictor_names=PREDICTOR_TYPE.ENVELOPE_ONSET, attention=ATTENTION_TYPE.IGNORED, model_type=MODEL_TYPE.BACKWARD)

TRF exists at /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/full_pooled_backward_attended_envelope.pickle, skipping.
TRF exists at /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/full_pooled_backward_ignored_envelope.pickle, skipping.
TRF exists at /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/full_pooled_backward_attended_envelope_onset.pickle, skipping.
TRF exists at /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/full_pooled_backward_ignored_envelope_onset.pickle, skipping.


In [4]:
def estimate_pooled_universal_trf_loo(subjects, predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.BACKWARD, padded=False, predictor_dir=FUGLSANG_PREDICTOR_DIR, loo=False):

    TRF_LAG_START        = -0.100
    TRF_LAG_END          =  1.000
    BASIS_FUNCTION_WIDTH =  0.050

    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]
    predictor_names = sorted(predictor_names, key=lambda p: p.value)

    new_model_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor_names, attention, model_type, generalised=GENERALISATION_TYPE.POOLED
    )
    suffix = "_padded" if padded else ""

    # ----------------------------------------------------------------
    # Early exit if all paths already exist
    # ----------------------------------------------------------------
    if loo:
        all_paths = [FUGLSANG_GENERAL_TRF_DIR / f"loocv_{s}_{new_model_name}.pickle" for s in subjects]
    else:
        all_paths = [FUGLSANG_GENERAL_TRF_DIR / f"full_{new_model_name}.pickle"]

    if all(p.exists() for p in all_paths):
        print(f"All TRFs already exist for {new_model_name}, skipping.")
        return

    # ----------------------------------------------------------------
    # Load all subjects
    # ----------------------------------------------------------------
    all_eeg        = {}
    all_predictors = {}

    for subject in subjects:
        print(f"  Loading {subject}...", end=' ')
        
        eeg_path = FUGLSANG_EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"
        if not eeg_path.exists():
            print(f"✗ Missing EEG, skipping.")
            continue

        eeg    = eelbrain.load.unpickle(eeg_path)
        failed = False
        predictors = []

        for p in predictor_names:
            predictor_name = helper_functions.get_attentional_predictor_name(p, attention, padded)
            predictor_path = predictor_dir / subject / f"{predictor_name}_concat.pickle"

            if not predictor_path.exists():
                print(f"✗ Missing predictor {predictor_path}, skipping.")
                failed = True
                break

            predictors.append(eelbrain.load.unpickle(predictor_path))

        if failed:
            continue

        all_eeg[subject]        = eeg
        all_predictors[subject] = predictors
        print(f"✓ EEG={eeg.shape}, predictor={predictors[0].shape}")

    if not all_eeg:
        print("No data loaded, aborting.")
        return

    print(f"\nLoaded {len(all_eeg)}/{len(subjects)} subjects successfully.")

    # ----------------------------------------------------------------
    # Determine folds and estimate
    # ----------------------------------------------------------------
    loaded_subjects = list(all_eeg.keys())
    folds = [(s, [o for o in loaded_subjects if o != s]) for s in loaded_subjects] if loo else [(None, loaded_subjects)]

    for held_out, train_subjects in tqdm(folds, desc='Estimating LOO TRFs'):
        if loo:
            save_path = FUGLSANG_GENERAL_TRF_DIR / f"loocv_{held_out}_{new_model_name}.pickle"
            print(f"\nLOO fold: holding out {held_out}, training on {len(train_subjects)} subjects")
        else:
            save_path = FUGLSANG_GENERAL_TRF_DIR / f"full_{new_model_name}.pickle"
            print(f"\nEstimating {new_model_name} TRF across {len(train_subjects)} subjects...")

        if save_path.exists():
            print(f"  Already exists, skipping.")
            continue

        eeg_list       = [all_eeg[s]        for s in train_subjects]
        predictor_list = [all_predictors[s] for s in train_subjects]

        if len(predictor_names) == 1:
            stimulus = [subj_preds[0] for subj_preds in predictor_list]
        else:
            stimulus = [
                [predictor_list[s][p] for s in range(len(eeg_list))]
                for p in range(len(predictor_names))
            ]

        if model_type == MODEL_TYPE.FORWARD:
            tmin, tmax = TRF_LAG_START, TRF_LAG_END
            trf = eelbrain.boosting(
                eelbrain.concatenate(eeg_list),
                eelbrain.concatenate(stimulus) if len(predictor_names) == 1 else [eelbrain.concatenate(s) for s in stimulus],
                tmin, tmax,
                error='l1', basis=BASIS_FUNCTION_WIDTH,
                partitions=5, test=1, selective_stopping=True
            )
        else:
            tmin, tmax = -TRF_LAG_END, -TRF_LAG_START
            trf = eelbrain.boosting(
                eelbrain.concatenate(stimulus) if len(predictor_names) == 1 else [eelbrain.concatenate(s) for s in stimulus],
                eelbrain.concatenate(eeg_list),
                tmin, tmax,
                error='l1', basis=BASIS_FUNCTION_WIDTH,
                partitions=5, test=1, selective_stopping=True
            )

        eelbrain.save.pickle(trf, save_path)
        print(f"  Saved → {save_path}")

In [5]:
# ─────────────────────────────────────────────
# Call sites
# ─────────────────────────────────────────────

# Pooled (original behaviour)
estimate_pooled_universal_trf_loo(SUBJECTS, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED)
estimate_pooled_universal_trf_loo(SUBJECTS, PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED)


# LOO
estimate_pooled_universal_trf_loo(SUBJECTS, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED, loo=True)

All TRFs already exist for pooled_backward_attended_envelope, skipping.
All TRFs already exist for pooled_backward_attended_envelope_onset, skipping.
All TRFs already exist for pooled_backward_attended_envelope, skipping.


In [6]:
estimate_pooled_universal_trf_loo(SUBJECTS, PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, loo=True)

  Loading S1... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S2... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S3... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S4... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S5... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S6... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S7... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S8... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S9... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S10... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S11... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S12... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S13... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S14... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S15... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S16... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S17... ✓ EEG=(64, 192000), predictor=(192000,)
  Loading S18... ✓ EEG=(64, 192000), pre

Estimating LOO TRFs:   0%|          | 0/18 [00:00<?, ?it/s]


LOO fold: holding out S1, training on 17 subjects


Estimating LOO TRFs:   6%|▌         | 1/18 [1:51:51<31:41:30, 6711.19s/it]

  Saved → /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/loocv_S1_pooled_backward_attended_envelope_onset.pickle

LOO fold: holding out S2, training on 17 subjects


...

  Saved → /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/loocv_S16_pooled_backward_attended_envelope_onset.pickle

LOO fold: holding out S17, training on 17 subjects


Estimating LOO TRFs:  94%|█████████▍| 17/18 [24:38:22<1:27:30, 5250.67s/it]

  Saved → /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/loocv_S17_pooled_backward_attended_envelope_onset.pickle

LOO fold: holding out S18, training on 17 subjects


Estimating LOO TRFs: 100%|██████████| 18/18 [27:27:07<00:00, 5490.39s/it]  

  Saved → /Users/sylvestereley/Data/Beyond-TRFs/Fuglsang/TRFs/self_computed/generalised/loocv_S18_pooled_backward_attended_envelope_onset.pickle
